**Required imports**

# Notebook 9 — Querying Hierarchical JSON Data for LLM Context

## What You Will Learn

Real-world JSON is rarely flat. APIs, configuration files, and survey exports are often deeply nested — with countries containing volunteers, each with attributes. This notebook teaches you to **query, filter, and slice nested JSON** using `JMESPath`, then pass the right slice as LLM context.

### Topics Covered

| Technique | Description |
|---|---|
| **JMESPath Queries** | A query language for JSON — filter, project, and slice nested structures |
| **Inner-Level Filtering** | Filter at the deepest level (individual records) while preserving outer keys |
| **Outer-Loop + Inner Query** | Loop over countries, apply JMESPath per country to retain country context |
| **List Comprehensions** | Extract specific fields from nested structures compactly |
| **LLM Context from JSON** | Serialize filtered results and pass to Groq for question answering |

### JMESPath Cheat Sheet (used in this notebook)

```python
# Filter volunteers in Jordan with High School education
"Jordan.volunteers[?education=='High School']"

# Per-country: get names of Urban residents
"volunteers[?location_type=='Urban'].name"

# Per-country: get age and sex of PhD holders
"volunteers[?education=='PhD'][].{age: age, sex: sex}"
```

### Skills You Will Build

- Load and navigate multi-level nested JSON structures
- Write JMESPath expressions to filter records at any nesting level
- Loop through outer dimensions to preserve outer-key context in results
- Extract and reshape JSON fields using list comprehensions
- Prepare filtered JSON as compact, relevant LLM context

> **Why it matters:** APIs, webhook payloads, and NoSQL exports are JSON. Being able to query them precisely means you can build RAG pipelines on top of any JSON data source — not just databases.

In [18]:
import json
import jmespath
from collections import Counter
from dotenv import load_dotenv
from hf_llm import hf_chat_completion

**Load the JSON File**  
Lod the JSON file which has survey data arranged in multi level nesting  

In [19]:
# Load JSON data from file
with open("survey_data.json", "r") as f:
    data = json.load(f)

>Key nesting is used as part of query.  
>Filter done at innesr most level (records)  


In [20]:
# Search for volunteers in Jordan with education "High School"
query = "Jordan.volunteers[?education=='High School']"
results = jmespath.search(query, data)

# Print the filtered records
for person in results:
    print(person)

{'name': 'John Daniel', 'age': 48, 'sex': 'Male', 'education': 'High School', 'annual_earning_usd': 82978.57, 'city': 'Smithshire', 'avg_weekly_work_hours': 49.5, 'location_type': 'Urban'}
{'name': 'Jennifer Thompson', 'age': 37, 'sex': 'Male', 'education': 'High School', 'annual_earning_usd': 95683.59, 'city': 'West Douglasville', 'avg_weekly_work_hours': 39.4, 'location_type': 'Urban'}
{'name': 'Christina Snow', 'age': 24, 'sex': 'Other', 'education': 'High School', 'annual_earning_usd': 38819.56, 'city': 'Andersonbury', 'avg_weekly_work_hours': 31.6, 'location_type': 'Rural'}
{'name': 'Jeremy Turner', 'age': 64, 'sex': 'Male', 'education': 'High School', 'annual_earning_usd': 30972.85, 'city': 'North Maria', 'avg_weekly_work_hours': 22.0, 'location_type': 'Urban'}
{'name': 'Eric Martinez', 'age': 29, 'sex': 'Male', 'education': 'High School', 'annual_earning_usd': 43343.48, 'city': 'Fisherfort', 'avg_weekly_work_hours': 58.4, 'location_type': 'Rural'}
{'name': 'Stephanie Flynn', 'ag

>Loop through at a level, to retain value of that field

In [21]:
# Extract names of urban residents per country
urban_names_per_country = {}

# Loop through the outer dimension
for country, info in data.items():
    #query to get names of volunteers living in urban areas
    query = "volunteers[?location_type=='Urban'].name"
    names = jmespath.search(query, info)
    urban_names_per_country[country] = names

urban_names_per_country


{'Jordan': ['Samuel Larson',
  'John Daniel',
  'Jennifer Thompson',
  'Kathleen Patton',
  'Michelle Rivers',
  'Sean Lane',
  'Jeffrey Heath',
  'Jeremy Turner',
  'Deborah Martinez',
  'Jared Martinez',
  'Brandon Gonzalez',
  'Susan Anderson',
  'Ashley Morgan',
  'Joshua Butler',
  'Stacy Hunter',
  'Brendan Miller',
  'Ms. Sherri Reilly',
  'Tracy Casey',
  'Ashley Spencer',
  'Jake Sanders',
  'Brittany Chen',
  'Christian Sampson',
  'David Park',
  'Daniel Johnson',
  'Stephanie Flynn',
  'Madison Moore',
  'Jonathan Alvarez',
  'Angela Campos',
  'Susan Ritter',
  'Kimberly Weber',
  'Michelle Ball',
  'Joshua Blanchard',
  'Kim White',
  'Kelsey Perkins',
  'Nathaniel Leonard',
  'Beth Carroll',
  'James Alexander',
  'Robin Stevens',
  'Austin Ellis',
  'Catherine Werner',
  'Laura Jones',
  'Kevin Arroyo',
  'Holly Burns',
  'Jeanne Mendoza',
  'Patrick Palmer',
  'Elizabeth Moore',
  'Erin Mcknight',
  'Raven Wheeler',
  'Allen Hooper',
  'Brian Holt',
  'Christian Long',

In [22]:
# Dictionary to hold results
phd_demographics_per_country = {}

# Apply JMESPath filter for each country
for country, info in data.items():
    result = jmespath.search("volunteers[?education=='PhD'][].{age: age, sex: sex}", info)
    if result:  # Only store countries with PhD individuals
        phd_demographics_per_country[country+' - PhD'] = result

phd_demographics_per_country

{'Jordan - PhD': [{'age': 28, 'sex': 'Female'},
  {'age': 42, 'sex': 'Female'},
  {'age': 45, 'sex': 'Male'},
  {'age': 24, 'sex': 'Male'},
  {'age': 27, 'sex': 'Other'},
  {'age': 27, 'sex': 'Other'},
  {'age': 51, 'sex': 'Female'},
  {'age': 48, 'sex': 'Other'},
  {'age': 61, 'sex': 'Female'},
  {'age': 29, 'sex': 'Female'},
  {'age': 58, 'sex': 'Male'},
  {'age': 63, 'sex': 'Male'},
  {'age': 45, 'sex': 'Other'},
  {'age': 59, 'sex': 'Male'},
  {'age': 37, 'sex': 'Female'},
  {'age': 63, 'sex': 'Other'}],
 'Moldova - PhD': [{'age': 25, 'sex': 'Other'},
  {'age': 31, 'sex': 'Male'},
  {'age': 42, 'sex': 'Other'},
  {'age': 22, 'sex': 'Other'},
  {'age': 54, 'sex': 'Male'},
  {'age': 19, 'sex': 'Male'},
  {'age': 38, 'sex': 'Male'},
  {'age': 49, 'sex': 'Other'},
  {'age': 67, 'sex': 'Female'},
  {'age': 58, 'sex': 'Female'},
  {'age': 39, 'sex': 'Male'},
  {'age': 69, 'sex': 'Other'},
  {'age': 60, 'sex': 'Male'},
  {'age': 24, 'sex': 'Male'},
  {'age': 23, 'sex': 'Other'},
  {'age':

**Retrived Data to LLM**  
The data retrieved from JSON is provided to LLM as context  
This provides the relevant data to respond to the prompt  


In [23]:
load_dotenv()
HF_Model = "google/flan-t5-base"

>Specific instuction for Response

In [24]:
R_Instr = "Using the context given, provide response to the user question or statement.\
            Context is provided in JSON format.\
            Provide a comprehensive response"

>The data is being retireved from JSON using list comprehension  


In [25]:
# User prompt
Prompt = "Who is more educated, in each country? Ladies or Gents?"

# Aggregate the counts in Python, rather than sending every record.
# The LLM then compares a handful of numbers instead of tallying ~250 records.
edu_counts_per_country = {}

for country, info in data.items():
    per_sex = {}

    # Derive the sex values present, instead of hardcoding them
    for sex in sorted (set (jmespath.search ("volunteers[].sex", info))):
        # Filter at the record level, project just the education field
        query = "volunteers[?sex=='"+sex+"'].education"
        per_sex[sex] = dict (Counter (jmespath.search (query, info)))

    edu_counts_per_country[country] = per_sex

Context = json.dumps (edu_counts_per_country)

print ("Context size :", len (Context), "chars")

# Invoke LLM with prompt and context
messages=[
    {
        "role": "system",
        "content": R_Instr
    },

    {
        "role": "user",
        "content":"Context : \n"+ Context
    },

    {
        "role": "user",
        "content": "Query : \n" + Prompt
    }
]
completion = hf_chat_completion(
    messages=messages,
    model="openai/gpt-oss-120b",
    # gpt-oss is a reasoning model -- reasoning tokens are drawn from this same
    # budget, so the default of 512 leaves nothing for the visible answer
    max_tokens=2048,
)

print (completion.choices[0].message.content)

Context size : 1765 chars
**Summary**

| Country   | Which gender has the larger “high‑education” total*? |
|-----------|------------------------------------------------------|
| Jordan    | **Males** (16 vs 14) |
| Moldova   | **Tie** – both 17 |
| Denmark   | **Males** (22 vs 16) |
| Gibraltar | **Tie** – both 16 |
| Morocco   | **Tie** – both 18 |

\*For this comparison I added together the numbers of people who hold the three highest‑level qualifications in the data set: **PhD**, **Master’s**, and **Bachelor’s**.  These three categories are the usual markers of “higher education”.  (All other categories – No schooling, High School, Associate Degree – were excluded from the “high‑education” count.)

---

### How the totals were calculated  

| Country | Female (PhD + Master’s + Bachelor’s) | Male (PhD + Master’s + Bachelor’s) |
|---------|----------------------------------------|--------------------------------------|
| **Jordan** | 6 + 4 + 4 = **14** | 5 + 1 + 10 = **16** |
| **Mol